Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\computadorNuevo2\\datosNarmax\\1pasos_gru_consumption.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,temp,zone1,zone2,zone3,hour,e
date,,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858,NaN
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858,NaN
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858,NaN
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858,NaN
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858,NaN


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Espacio de búsqueda

In [5]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [6]:
futuros = 1
pasados  = 12

In [7]:
datosX = []
datosY = []
for i in range(pasados, len(datos) - futuros + 1):
  datosX.append(datos.iloc[i-pasados:i, 0:datos.shape[1]])
  datosY.append(datos.iloc[i+futuros-1:i+futuros, 1])


In [8]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (52404, 12, 6)
Dimensiones de Y: (52404, 1)


In [9]:
print(datosX[0])

[[-2.05135573  0.15144479 -0.85166941  0.03349964 -1.66085756         nan]
 [-2.07481311 -0.43307937 -0.21109699  0.01650173 -1.66085756         nan]
 [-2.0911524  -0.52770864 -0.28379117 -0.05506842 -1.66085756         nan]
 [-2.12221321 -0.65164785 -0.41118591 -0.17405378 -1.66085756         nan]
 [-2.15456823 -0.77474965 -0.50763164 -0.2447293  -1.66085756         nan]
 [-2.16556893 -0.87272862 -0.59759968 -0.29303915 -1.66085756         nan]
 [-2.19986525 -0.95898362 -0.68109001 -0.32166721 -1.51634012         nan]
 [-2.22332263 -1.03518949 -0.74658674 -0.39681586 -1.51634012         nan]
 [-2.19387957 -1.12730648 -0.83223632 -0.46391287 -1.51634012         nan]
 [-2.22413151 -1.19597551 -0.88909611 -0.49969795 -1.51634012         nan]
 [-2.22008713 -1.24873342 -0.98842083 -0.52385287 -1.51634012         nan]
 [-2.22736701 -1.29730419 -1.03232523 -0.5614272  -1.51634012         nan]]


Se dividen nuevamente los conjuntos de datos

In [10]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (36682, 12, 6)
Las dimensiones de testX son:  (10533, 12, 6)
Las dimensiones de valX son:  (5189, 12, 6)


In [11]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (36682, 1)
Las dimensiones de testY son:  (10533, 1)
Las dimensiones de valY son:  (5189, 1)


Se crean métricas para medir desempeño

In [12]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [13]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [14]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(testX.shape[1], testX.shape[2])))
    if (params['layers'] == 1):
      model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(GRU(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(GRU(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(testX, testY, epochs=128,
                        validation_split=0.3,
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [15]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

58/58 - 15s - 254ms/step - ia: 0.3364 - loss: 1.3852 - mae: 0.9271 - rmse: 1.1735 - smape: 1.3727 - val_ia: 0.1476 - val_loss: 0.6576 - val_mae: 0.6853 - val_rmse: 0.8084 - val_smape: 1.0387

Epoch 2/128                                           

58/58 - 1s - 14ms/step - ia: 0.3108 - loss: 1.1607 - mae: 0.8575 - rmse: 1.0751 - smape: 1.4085 - val_ia: 0.2312 - val_loss: 0.7074 - val_mae: 0.7016 - val_rmse: 0.8363 - val_smape: 1.1379

Epoch 3/128                                           

58/58 - 1s - 16ms/step - ia: 0.2770 - loss: 1.0641 - mae: 0.8260 - rmse: 1.0292 - smape: 1.4693 - val_ia: 0.2812 - val_loss: 0.7798 - val_mae: 0.7370 - val_rmse: 0.8762 - val_smape: 1.2951

Epoch 4/128                                           

58/58 - 1s - 19ms/step - ia: 0.2598 - loss: 0.9921 - mae: 0.8061 - rmse: 0.9942 - smape: 1.4792 - val_ia: 0.2997 - val_loss: 0.8538 - val_mae: 0.7749 - val_rmse: 0.9156 - val_smape: 1.4961

Epoch 5/128   

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                     

461/461 - 27s - 58ms/step - ia: 0.9182 - loss: 0.0354 - mae: 0.1092 - rmse: 0.1445 - smape: 0.2777 - val_ia: 0.6569 - val_loss: 0.0125 - val_mae: 0.0881 - val_rmse: 0.1012 - val_smape: 0.1860

Epoch 2/128                                                                     

461/461 - 11s - 24ms/step - ia: 0.9683 - loss: 0.0037 - mae: 0.0443 - rmse: 0.0582 - smape: 0.1381 - val_ia: 0.7503 - val_loss: 0.0054 - val_mae: 0.0547 - val_rmse: 0.0665 - val_smape: 0.1264

Epoch 3/128                                                                     

461/461 - 11s - 24ms/step - ia: 0.9697 - loss: 0.0033 - mae: 0.0420 - rmse: 0.0546 - smape: 0.1373 - val_ia: 0.8021 - val_loss: 0.0036 - val_mae: 0.0412 - val_rmse: 0.0531 - val_smape: 0.1095

Epoch 4/128                                                                     

461/461 - 11s - 24ms/step - ia: 0.9717 - loss: 0.0029 - mae: 0.0392 - rmse: 0.0516 - smape: 0

...

Epoch 1/128                                                                         

461/461 - 13s - 28ms/step - ia: 0.3210 - loss: 0.9852 - mae: 0.8072 - rmse: 0.9770 - smape: 1.4058 - val_ia: 0.2113 - val_loss: 0.6378 - val_mae: 0.6812 - val_rmse: 0.7066 - val_smape: 1.3824

Epoch 2/128                                                                         

461/461 - 9s - 20ms/step - ia: 0.5518 - loss: 0.4865 - mae: 0.5497 - rmse: 0.6792 - smape: 1.0357 - val_ia: 0.2896 - val_loss: 0.2791 - val_mae: 0.4605 - val_rmse: 0.4832 - val_smape: 0.8815

Epoch 3/128                                                                         

461/461 - 6s - 13ms/step - ia: 0.6924 - loss: 0.2789 - mae: 0.4210 - rmse: 0.5199 - smape: 0.8081 - val_ia: 0.3101 - val_loss: 0.2261 - val_mae: 0.4159 - val_rmse: 0.4350 - val_smape: 0.8170

Epoch 4/128                                                                         

461/461 - 10s - 22ms/step - ia: 0.7298 - loss: 0.2180 - mae: 0.3703 - rmse: 0.4

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                            

922/922 - 22s - 24ms/step - ia: 0.1827 - loss: 0.9803 - mae: 0.8216 - rmse: 0.9703 - smape: 1.7474 - val_ia: 0.0945 - val_loss: 1.4709 - val_mae: 1.0159 - val_rmse: 1.0283 - val_smape: 1.8776

Epoch 2/128                                                                            

922/922 - 9s - 10ms/step - ia: 0.1809 - loss: 0.9603 - mae: 0.8138 - rmse: 0.9608 - smape: 1.7395 - val_ia: 0.0948 - val_loss: 1.4547 - val_mae: 1.0106 - val_rmse: 1.0230 - val_smape: 1.8779

Epoch 3/128                                                                            

922/922 - 7s - 7ms/step - ia: 0.1834 - loss: 0.9536 - mae: 0.8114 - rmse: 0.9573 - smape: 1.7463 - val_ia: 0.0951 - val_loss: 1.4387 - val_mae: 1.0052 - val_rmse: 1.0176 - val_smape: 1.8783

Epoch 4/128                                                                            

922/922 - 7s - 8ms/step - ia: 0.1834 - loss: 0.9434 - mae: 0.8070 - 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                            

58/58 - 5s - 93ms/step - ia: 0.3230 - loss: 1.1397 - mae: 0.8416 - rmse: 1.0398 - smape: 1.3817 - val_ia: 0.4010 - val_loss: 0.7253 - val_mae: 0.7293 - val_rmse: 0.8423 - val_smape: 1.5239

Epoch 2/128                                                                            

58/58 - 1s - 16ms/step - ia: 0.4481 - loss: 0.5694 - mae: 0.6037 - rmse: 0.7512 - smape: 1.2088 - val_ia: 0.5357 - val_loss: 0.3915 - val_mae: 0.5427 - val_rmse: 0.6177 - val_smape: 1.0885

Epoch 3/128                                                                            

58/58 - 1s - 15ms/step - ia: 0.6274 - loss: 0.3403 - mae: 0.4594 - rmse: 0.5806 - smape: 0.9221 - val_ia: 0.6604 - val_loss: 0.2505 - val_mae: 0.4367 - val_rmse: 0.4930 - val_smape: 0.8933

Epoch 4/128                                                                            

58/58 - 1s - 15ms/step - ia: 0.7093 - loss: 0.2547 - mae: 0.3960 - rmse: 0

In [16]:
print(best)

{'activation': 3, 'batch': 1, 'dropout': 0.0, 'layers': 4.0, 'learning_rate': 0.00027101707182655693, 'units': 4}


In [17]:
# {'activation': 3, 'batch': 1, 'dropout': 0.0, 'layers': 4.0, 'learning_rate': 0.00027101707182655693, 'units': 4}